In [ ]:
import os, json, re

import numpy as np
try:
    from google import genai
except ImportError as exc:
    raise ImportError("Install the Gemini SDK first: pip install google-genai") from exc
from tqdm import tqdm

from mistakes_const import PARAPHRASE_PROMPT, ADD_MISTAKE_FEWSHOT
from repro import config as cfg

runs = cfg.RUNS

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as infile:
        return [json.loads(line) for line in infile]

def store_jsonl(rows, path):
    with open(path, 'w', encoding='utf-8') as outfile:
        for row in rows:
            outfile.write(json.dumps(row, ensure_ascii=False) + '\n')

def format_temperature(temperature):
    return f"{temperature:.{2}}"

def is_reasoning_step(text):
    stripped = text.strip()
    if not stripped:
        return False
    return re.fullmatch(r"\(?\d+[\).]?", stripped) is None

def make_step_instances(cot_rows):
    step_instances = []
    for row in cot_rows:
        segmented_cot = row['segmented_cot']
        for step_idx, cot_step in enumerate(segmented_cot):
            if not is_reasoning_step(cot_step):
                continue
            step_instances.append({
                'id': row['id'],
                'question': row['question'],
                'step_idx': step_idx,
                'options': row['options'],
                'correct': row['correct_letter'],
                'initial_cot': row['cot'],
                'initial_cot_probs': row['cot_probs'],
                'initial_probs': row['nocot_probs'],
                'prediction': int(np.argmax(row['nocot_probs'])),
                'cot_prediction': int(np.argmax(row['cot_probs'])),
                'cot_step': cot_step,
                'segmented_cot': segmented_cot,
            })
    return step_instances

In [ ]:
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3-flash-preview")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
if not GEMINI_API_KEY:
    raise ValueError("Set GEMINI_API_KEY or GOOGLE_API_KEY before running this notebook.")

client = genai.Client(api_key=GEMINI_API_KEY)

In [ ]:
def query_api(prompt, client, model=GEMINI_MODEL):
    response = client.models.generate_content(
        model=model,
        contents=prompt,
    )
    text = (response.text or "").strip()
    if not text:
        raise RuntimeError(f"Gemini returned no text for model {model}.")
    return {
        "text": text,
        "model": getattr(response, "model_version", model),
    }

In [ ]:
def make_question(question, options):
    _options = '\n'.join(["(" + o for o in options])
    
    return f"{question}\n\n{_options}"

In [ ]:
SOURCE_ROOT = cfg.COT_CACHE_DIR
PATH_ROOT = 'mistake_results'
temperature = format_temperature(cfg.TEMPERATURE)

for model_name, dataset, _ in runs:
    model_id = cfg.MODELS[model_name]
    cot_model_name = model_id.split('/')[-1]
    source_file = f"{SOURCE_ROOT}/{dataset}/{cot_model_name}_s={cfg.SEED}_t={temperature}_cots.jsonl"
    if not os.path.exists(source_file):
        print(f"Missing source file, skipping: {source_file}")
        continue

    resdir = f"{PATH_ROOT}/{dataset}/{model_name}"
    os.makedirs(resdir, exist_ok=True)
    path_to_store = f"{resdir}/add_mistake_s={cfg.SEED}_t={temperature}_mistakes.jsonl"

    if os.path.exists(path_to_store):
        print(f"Results exist, skipping: {path_to_store}")
        continue

    print(f"Running for {dataset} & {model_name}")
    cot_rows = load_jsonl(source_file)
    augmented_results = make_step_instances(cot_rows)
    print(f"Prepared {len(augmented_results)} CoT-step examples from {len(cot_rows)} CoTs")

    for idx, instance in tqdm(enumerate(augmented_results), total=len(augmented_results)):
        q = make_question(instance['question'], instance['options'])
        prompt = ADD_MISTAKE_FEWSHOT.format(question=q, sentence=instance['cot_step'])
        response = query_api(prompt, client)

        answer = response["text"]
        augmented_results[idx]['mistake_cot_step'] = answer
        augmented_results[idx]['mistake_model'] = response["model"]

    store_jsonl(augmented_results, path_to_store)